In [73]:
# reload the modules
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
import numpy as np
from minitorch.tensor.tensor import Tensor
from minitorch.nn.layers import Linear, Sequential, Dropout
from minitorch.activations.activations import ReLU, GELU, Tanh, Sigmoid, Softmax
from minitorch.dataloaders.dataloader import DataLoader, TensorDataset
from minitorch.losses.losses import BinaryCrossEntropy
from minitorch.labs.loan_default.src.components.data_transformation import DataTransformation
from minitorch.labs.loan_default.src.pipeline.model_pipeline import LoanDefaultPredictor
from minitorch.optimizers.optim import AdamW, SGD
from minitorch.train.training import Trainer, CosineSchedule

Data Ingestion and Transformation


In [12]:
# Ingestion and transform the data
data_transformer = DataTransformation()
data_transformer.train_set = data_transformer.train_set[:1000]
data_transformer.test_set = data_transformer.test_set[:100]

train_arr, test_arr = data_transformer.initiate_data_transformation()

🧪 Ingesting data...
📥 Loading dataset from C:\Users\User\Desktop\Loan_Default.csv...
✅ Loaded dataset successfully.
🔀 Splitting dataset into training and testing sets...
✅ Split dataset successfully.
💾 Saving dataset to artifacts directory...
📁 Checking if directory exists ...
📁 Directory already exists at c:\Users\User\Desktop\babytorch\minitorch\labs\loan_default\artifacts.
✅ Data already ingested.
✅ Found existing training and testing datasets. Returning their paths.
🧪 Getting the preprocessing Object ...
✍️ Applying transformation to training datasets ...
✍️ Applying transformation to testing datasets ...
Applying transformation to the response variable ...
✅ Done with the transformatiom 
🪡🧵Concatenating the features and the targets back together ...
✅✅ Done with the concatination and the data transformatiom.
(118936, 49) (29734, 49)


In [13]:
# # get the features and target and convert them to tensors
features, target = train_arr[:, 1:-2], train_arr[:, -1]
features, target = Tensor(features, requires_grad=True), Tensor(target, requires_grad=True)

# # create the data loader
ds = TensorDataset(features, target)
train_loader = DataLoader(ds, batch_size=32, shuffle=False)

In [75]:
# create the model
MAX_EPOCHS = 2
model = LoanDefaultPredictor(in_features=features.shape[1], out_features=1, drop_out_p=0.0)
loss_fn = BinaryCrossEntropy()
optimizer = SGD(model.parameters(), lr=0.003, momentum=0.9, weight_decay=0.1)
# optimizer = AdamW(model.parameters(), lr=0.03, weight_decay=0.1)
scheduler = CosineSchedule(max_lr=0.03, min_lr = 0.003, total_epochs=MAX_EPOCHS)
trainer = Trainer(model=model,
                loss_fn=loss_fn,
                optimizer=optimizer,
                scheduler=scheduler,
                clip_gradients=True)



In [77]:
STEPS = 1
print()
print('Full model training...\n')
for epoch in range(MAX_EPOCHS):
    loss = trainer.train_epoch(train_loader)
        
    if (epoch + 1) % STEPS == 0:
        print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Average Loss: {loss:.4f}')
        print(f'Learning Rate at epoch {epoch+1}: {trainer.scheduler.get_lr(epoch):.6f}\n')
        
        # for p in model.parameters():
        #     print(f'Parameter: {p.data.flatten()[:5]}')
        print('\n')


Full model training...



ValueError: setting an array element with a sequence.

In [19]:
from minitorch.nn.layers import Linear

In [78]:
in_features = features.shape[1]
out_features = 5
linear = Linear(in_features, out_features)
act = Tanh()
out = act(linear(features))

In [79]:
out.backward()

In [80]:
linear.weight.grad

array([[ 1.03319128e+03,  3.61971313e+02,  1.12509424e+04,
         9.65051758e+03, -1.84204980e+03],
       [ 2.47586060e+02,  9.14606738e+03,  1.13701406e+04,
         1.51835781e+04, -1.01359949e+03],
       [-3.13536890e+03,  5.12586328e+03,  8.50927344e+03,
        -1.06574746e+04,  6.55429138e+02],
       [ 1.32278320e+03, -3.73738623e+03,  3.25945459e+03,
         6.91327942e+02,  2.56336401e+03],
       [-1.16899182e+03, -5.34133008e+03, -7.16608447e+03,
        -1.41191670e+04, -1.73284619e+03],
       [-3.03510693e+03, -1.65071436e+03, -2.73272144e+03,
        -1.11646699e+04, -3.19354541e+03],
       [ 2.39302808e+03, -3.11569580e+03,  1.78438586e+03,
         8.23828906e+03,  2.69870874e+03],
       [-4.57395111e+02,  6.66520386e+02, -9.44258423e+02,
         3.16640967e+03,  8.96129990e+01],
       [-9.56350464e+02, -3.73389320e+01, -5.37723975e+03,
         7.32136768e+03,  2.92335400e+03],
       [ 6.91122412e+03,  6.04184619e+03,  4.38596289e+03,
         4.99761377e+03

In [69]:
w = Tensor([0.5], requires_grad=True)
x = Tensor([2.0])
out = w * x
out.backward()
print(w.grad) # Should be 2.0


[2.]
